# SpaceL2_Flow: a quick demonstration

Two Roman WFI Level 2 files (`_cal.asdf`, same filter) go in. sfft reads no file: `flow_inputs` from `read_roman_l2.py` (next to this
notebook) reads both with `roman_datamodels` and gives their arrays and keywords to `SpaceL2_Flow`, which prepares both, resamples the
reference onto the science exposure's grid, matches them with SFFT, and writes the difference, the score image and the variance (`diff.fits`) with the other
products into `OUTPUT_DIR`. The notebook then shows the full-frame difference and score, lists the candidates above a score threshold and
shows a stamp of each.

Run this notebook from the `examples/` folder (so that `read_roman_l2` can be imported), set the two files in the settings cell and
run all cells. A GPU is used when one works here (`BACKEND = "auto"`); on a CPU a full frame
takes a few minutes. The plots read `OUTPUT_DIR/diff.fits`, so they can be rerun without running the flow again.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.style.use('default')
latex_font_style = {
    "text.usetex": False,     # False: no LaTeX needed to run this notebook (True needs a latex binary)
    "font.family": "serif",
    "font.sans-serif": "sans-serif",
    "font.serif": ['Palatino', 'Palatino Linotype', 'URW Palladio L', 'DejaVu Serif'],
    "text.latex.preamble": ''
}
plt.rcParams.update(latex_font_style)

display(HTML("""
    <style>
        .cell-output-ipywidget-background {
        background-color: transparent !important;
        }
        body {
            --vscode-font-family: "Symbola"
        }
    </style>
    """)
)

import os
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval
from scipy import ndimage

## Settings

In [ ]:
SCIENCE_FILE = "path/to/your/l2/file/_cal.asdf"    # Level 2 file: the science exposure (its grid is the products' grid)
REFERENCE_FILE = "path/to/your/l2/file/_cal.asdf"  # Level 2 file: the reference, same filter
CACHE_DIR = "path/to/your/cache"                  # folder for the PSF models; None: ~/.cache/sfft_romanl2
OUTPUT_DIR = "spaceflow_demo_output"              # where the products are written

BACKEND = "auto"                                  # "auto" (GPU when one works here), "gpu" or "cpu"
CROP_PX = None                                    # None: full frame; e.g. 1024 for a quick look at a 1024 x 1024 part
DETECT_SIGMA = 5.0                                # detection threshold on the score image
N_STAMPS = 8                                      # candidates shown as stamps

## Run the flow

The two files are read here (`flow_inputs`), then one call does everything else; `run()` returns the record of the run (also written as `record.json`).

In [ ]:
from sfft.spaceflow import SpaceL2_Flow
from read_roman_l2 import flow_inputs     # sfft reads no file: this reads both and gives their arrays and keywords

inputs, keywords = flow_inputs(SCIENCE_FILE, REFERENCE_FILE)
flow = SpaceL2_Flow(*inputs, output_dir=OUTPUT_DIR, backend=BACKEND, crop_px=CROP_PX, cache_dir=CACHE_DIR, **keywords)
record = flow.run()

sfft_fit = record["sfft"]
print(f"done in {record['timings']['total']:.0f} s on the {record['backend']['used'].upper()}")
print(f"SFFT: flux scale {sfft_fit.get('fscal'):.5f}, {sfft_fit.get('fit_pixels')} pixels in the fit")
print("products:", sorted(os.listdir(OUTPUT_DIR)))

In [ ]:
with fits.open(os.path.join(OUTPUT_DIR, "diff.fits")) as hdul:
    diff = hdul["DIFF"].data.astype(float)          # science - matched reference
    score = hdul["SCORE"].data.astype(float)        # detection significance (sigma)
    sci_m = hdul["SCI_M"].data.astype(float)        # the two images as subtracted (matched)
    ref_m = hdul["REF_M"].data.astype(float)
    saturated = hdul["SATURATED"].data.astype(bool)
    wcs = WCS(hdul["DIFF"].header)
    unit = hdul["DIFF"].header.get("BUNIT", "")
ny, nx = diff.shape
print(f"{nx} x {ny} pixels, difference in {unit}")


def zlim(a):
    """DS9 zscale limits, computed on the whole image."""
    return ZScaleInterval(contrast=0.25).get_limits(a[np.isfinite(a)])

## The full frame

Science (matched), difference and score. The first two are shown in grayscale with zscale limits; the score is a derived map (sigma),
centred on zero. A full frame has many more pixels than the figure, so these panels are drawn reduced; the stamps below keep every pixel.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6), constrained_layout=True)
for ax, img, title in ((axes[0], sci_m, "science (matched)"), (axes[1], diff, "difference")):
    lo, hi = zlim(img)
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi, origin="lower")
    ax.set_title(title)
im = axes[2].imshow(score, cmap="RdBu_r", vmin=-DETECT_SIGMA, vmax=DETECT_SIGMA, origin="lower")
axes[2].set_title("score")
fig.colorbar(im, ax=axes[2], shrink=0.8, label="score (sigma)")
for ax in axes:
    ax.set_xlabel("x (pixel)")
axes[0].set_ylabel("y (pixel)")
plt.show()

## Candidates

Connected pixels with score above `DETECT_SIGMA` form one candidate, placed at its highest score. No vetting is done here: the list can
include leftovers at bright or saturated stars (the `near_saturated` column flags those within 5 pixels of a saturated pixel).

In [ ]:
above = np.isfinite(score) & (score > DETECT_SIGMA)
labels, n = ndimage.label(above, structure=np.ones((3, 3)))
index = np.arange(1, n + 1)
peak_pos = ndimage.maximum_position(np.where(above, score, -np.inf), labels, index)
peak = ndimage.maximum(score, labels, index)
npix = ndimage.sum(above, labels, index)
near_sat = ndimage.binary_dilation(saturated, iterations=5)

cands = []
for (y, x), s, m in zip(peak_pos, peak, npix):
    ra, dec = wcs.pixel_to_world_values(x, y)
    cands.append(dict(x=int(x), y=int(y), ra=float(ra), dec=float(dec), peak_score=float(s), n_pixels=int(m),
                      near_saturated=bool(near_sat[y, x])))
cands.sort(key=lambda c: -c["peak_score"])
print(f"{len(cands)} candidates above {DETECT_SIGMA:g} sigma")
print(f"{'#':>3} {'x':>6} {'y':>6} {'RA (deg)':>11} {'Dec (deg)':>10} {'score':>7} {'pixels':>7}  near saturated")
for i, c in enumerate(cands, 1):
    print(f"{i:3d} {c['x']:6d} {c['y']:6d} {c['ra']:11.6f} {c['dec']:10.6f} {c['peak_score']:7.1f} {c['n_pixels']:7d}  {c['near_saturated']}")

fig, ax = plt.subplots(figsize=(8, 8))
lo, hi = zlim(diff)
ax.imshow(diff, cmap="gray", vmin=lo, vmax=hi, origin="lower")
for i, c in enumerate(cands, 1):
    ax.add_patch(plt.Circle((c["x"], c["y"]), 0.012 * max(nx, ny), fill=False, color="orangered", lw=1.2))
    ax.annotate(str(i), (c["x"], c["y"]), xytext=(6, 6), textcoords="offset points", color="orangered", fontsize=9)
ax.set_title(f"difference with the {len(cands)} candidates above {DETECT_SIGMA:g} sigma")
ax.set_xlabel("x (pixel)")
ax.set_ylabel("y (pixel)")
plt.show()

## Stamps

For the strongest candidates: science and reference as subtracted (matched), difference and score, 41 x 41 pixels around the candidate.
Grayscale limits are the whole image's zscale limits; every pixel is kept.

In [ ]:
def cutout(img, x, y, half=20):
    out = np.full((2 * half + 1, 2 * half + 1), np.nan)
    y0, y1, x0, x1 = max(y - half, 0), min(y + half + 1, ny), max(x - half, 0), min(x + half + 1, nx)
    out[y0 - (y - half):y1 - (y - half), x0 - (x - half):x1 - (x - half)] = img[y0:y1, x0:x1]
    return out


def imshow_pixels(ax, a, min_side=600, **kw):
    """Every pixel kept: each repeated k x k times and drawn without interpolation; extent in the stamp's own pixels."""
    h, w = a.shape
    k = max(1, -(-min_side // max(h, w)))
    a = np.repeat(np.repeat(a, k, axis=0), k, axis=1)
    return ax.imshow(a, interpolation="none", origin="lower", extent=(-0.5, w - 0.5, -0.5, h - 0.5), **kw)


shown = cands[:N_STAMPS]
if shown:
    limits = {name: zlim(img) for name, img in (("sci", sci_m), ("ref", ref_m), ("diff", diff))}
    fig, axes = plt.subplots(len(shown), 4, figsize=(9, 2.3 * len(shown)), squeeze=False, constrained_layout=True)
    for row, (i, c) in zip(axes, enumerate(shown, 1)):
        for ax, (name, img) in zip(row, (("sci", sci_m), ("ref", ref_m), ("diff", diff))):
            imshow_pixels(ax, cutout(img, c["x"], c["y"]), cmap="gray", vmin=limits[name][0], vmax=limits[name][1])
        im = imshow_pixels(row[3], cutout(score, c["x"], c["y"]), cmap="RdBu_r", vmin=-2 * DETECT_SIGMA, vmax=2 * DETECT_SIGMA)
        for ax in row:
            ax.add_patch(plt.Circle((20, 20), 6, fill=False, color="orangered", lw=1))
            ax.set_xticks([])
            ax.set_yticks([])
        row[0].set_ylabel(f"#{i}  score {c['peak_score']:.0f}")
    for ax, title in zip(axes[0], ("science (matched)", "reference (matched)", "difference", "score")):
        ax.set_title(title)
    fig.colorbar(im, ax=axes[:, 3], shrink=0.5, label="score (sigma)")
    plt.show()